In [ ]:
from pathlib import Path
import pandas as pd
import birdnet
from birdnet import SpeciesPredictions, predict_species_within_audio_file

# define the root folder containing the audio files
folder_path = Path("data/")

# loop through every .wav file in the folder and its subdirectories
for audio_path in folder_path.rglob("*.wav"):
    # extract the date and hour from the filename
    filename_stem = audio_path.stem  # e.g., '20240607_050000'
    year = int(filename_stem[:4])
    month = int(filename_stem[4:6])
    date = int(filename_stem[6:8])
    hour = int(filename_stem[9:11])  # Hour is after the underscore
    
    # predict species within the audio file
    predictions = SpeciesPredictions(predict_species_within_audio_file(audio_path))
    
    # Build rows from predictions
    rows = []
    for time_range, species_dict in predictions.items():
        for species, confidence in species_dict.items():
            scientific_name = species.split('_', 1)[0]  # Only take the part before "_"
            rows.append((time_range[0], time_range[1], scientific_name, confidence))
    
    # Create the DataFrame and add the extra columns
    df = pd.DataFrame(rows, columns=["start_time", "end_time", "scientific_name", "confidence"])
    df = df[df['confidence'] > 0.5] # Probability level can be adjusted
    df['scientific_name'] = df['scientific_name'].str.lower()
    df['start_time'] = df['start_time'].astype(int)
    df['end_time'] = df['end_time'].astype(int)
    
    # add extra columns 
    df['year'] = year
    df['month'] = month
    df['date'] = date
    df['hour'] = hour
    df['ID'] = audio_path.parent.name  # Folder name as ID

    # group by all key columns so extra columns are retained
    df_grouped = df.groupby(
        ['start_time', 'end_time', 'year', 'month', 'date', 'hour', 'ID'],
        as_index=False
    ).agg(
        scientific_name=('scientific_name', lambda x: ', '.join(x)),
        confidence=('confidence', lambda x: list(x))
    )
    
    # split the aggregated scientific names and confidence lists into separate columns
    scientific_names_split = df_grouped['scientific_name'].str.split(', ', expand=True)
    num_species = scientific_names_split.shape[1]
    confidence_split = pd.DataFrame(df_grouped['confidence'].to_list(),
                                    columns=[f'confidence_{i+1}' for i in range(num_species)])
    
    # create new columns for each species and its corresponding confidence
    for i in range(num_species):
        df_grouped[f'scientific_name_{i+1}'] = scientific_names_split[i]
        df_grouped[f'confidence_{i+1}'] = confidence_split[f'confidence_{i+1}']
    
    # drop the intermediate aggregated columns
    df_grouped = df_grouped.drop(columns=['scientific_name', 'confidence'])
    
    # round all confidence columns to 3 decimal places
    confidence_columns = [col for col in df_grouped.columns if col.startswith('confidence')]
    df_grouped[confidence_columns] = df_grouped[confidence_columns].round(3)
    
    # save the DataFrame to a .csv file in the same folder as the original .wav file
    csv_filename = audio_path.stem + '.csv'
    df_grouped.to_csv(audio_path.parent / csv_filename, index=False)
    print(f"Processed and saved: {csv_filename}")


Predicting species: 100%|██████████| 900/900 [00:19<00:00, 45.51s/s] 


Processed and saved: 20240603_050000.csv


Predicting species: 100%|██████████| 900/900 [00:23<00:00, 38.65s/s] 


Processed and saved: 20240603_060000.csv
